In [1]:
import pandas as pd
from sklearn import preprocessing
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.linear_model import SGDClassifier
import numpy as np

np.random.seed(3407)


In [2]:
df = pd.read_csv("seattle-weather.csv")
df


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,rain
1457,2015-12-28,1.5,5.0,1.7,1.3,rain
1458,2015-12-29,0.0,7.2,0.6,2.6,fog
1459,2015-12-30,0.0,5.6,-1.0,3.4,sun


In [3]:
le = preprocessing.LabelEncoder()
df["weather"] = le.fit_transform(df["weather"])
df


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,0
1,2012-01-02,10.9,10.6,2.8,4.5,2
2,2012-01-03,0.8,11.7,7.2,2.3,2
3,2012-01-04,20.3,12.2,5.6,4.7,2
4,2012-01-05,1.3,8.9,2.8,6.1,2
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,2
1457,2015-12-28,1.5,5.0,1.7,1.3,2
1458,2015-12-29,0.0,7.2,0.6,2.6,1
1459,2015-12-30,0.0,5.6,-1.0,3.4,4


In [4]:
scaler = preprocessing.StandardScaler()
X_train = df.iloc[0:1061].drop(["weather", "date"], axis=1)
X_train = pd.DataFrame(data=scaler.fit_transform(X_train), columns=X_train.columns)
X_train


,precipitation,temp_max,temp_min,wind
0,-0.470248,-0.479315,-0.626347,0.990984
1,1.270699,-0.779524,-1.062471,0.854195
2,-0.342472,-0.629419,-0.190223,-0.650493
3,2.772066,-0.561190,-0.507404,0.990984
4,-0.262612,-1.011503,-1.062471,1.948513
...,...,...,...,...
1056,-0.390388,-0.943274,-0.289342,0.990984
1057,1.430419,-0.479315,-0.507404,1.264564
1058,-0.262612,-0.629419,-0.745290,0.375431
1059,2.452626,-0.329211,0.245901,0.854195


In [5]:
y_train = df.iloc[0:1061]["weather"]
y_train


0       0
1       2
2       2
3       2
4       2
       ..
1056    2
1057    2
1058    2
1059    2
1060    2
Name: weather, Length: 1061, dtype: int64

In [6]:
X_test = df.iloc[1061:].drop(["weather", "date"], axis=1)
X_test = pd.DataFrame(data=scaler.transform(X_test), columns=X_test.columns)
X_test


,precipitation,temp_max,temp_min,wind
0,0.056828,-0.260982,0.701850,2.290487
1,5.008144,-0.479315,-0.963352,1.743328
2,0.104744,-1.625566,-2.469963,1.401354
3,-0.470248,-1.843899,-2.588906,0.785800
4,-0.470248,-1.625566,-2.251901,-0.718888
...,...,...,...,...
395,0.903343,-1.625566,-1.280534,-0.240123
396,-0.230668,-1.543691,-1.280534,-1.334442
397,-0.470248,-1.243482,-1.498596,-0.445308
398,-0.470248,-1.461816,-1.815777,0.101851


In [7]:
y_test = df.iloc[1061:]["weather"]
y_test


1061    2
1062    2
1063    3
1064    4
1065    4
       ..
1456    2
1457    2
1458    1
1459    4
1460    4
Name: weather, Length: 400, dtype: int64

In [8]:
clf = SGDClassifier(n_jobs=-1, max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


0.725

In [ ]:
parameters = {
    "loss": (
        "hinge",
        "log_loss",
        "log",
        "modified_huber",
        "squared_hinge",
        "perceptron",
    ),
    "penalty": ("l2", "l1", "elasticnet"),
    "alpha": (0.1, 0.01, 0.001, 0.0001, 0.00001),
    "tol": (0.1, 0.01, 0.001, 0.0001, 0.00001),
    "epsilon": (0.1, 0.01, 0.001, 0.0001, 0.00001),
    "learning_rate": ("constant", "optimal", "invscaling", "adaptive"),
}
gs = GridSearchCV(SGDClassifier(max_iter=2000), parameters, n_jobs=-1)
gs.fit(X_train, y_train)


In [10]:
gs.best_params_


{'alpha': 0.001,
 'epsilon': 1e-05,
 'learning_rate': 'optimal',
 'loss': 'modified_huber',
 'penalty': 'l1',
 'tol': 0.001}

In [11]:
pred = gs.predict(X_test)
accuracy_score(y_test, pred)


0.83

In [12]:
parameters = {
    "C": (0.01, 0.1, 1, 10, 100, 1000),
    "kernel": ("linear", "poly", "rbf", "sigmoid"),
    "gamma": ("scale", "auto"),
}
gs = GridSearchCV(SVC(), parameters, n_jobs=-1)
gs.fit(X_train, y_train)


GridSearchCV(estimator=SVC(), n_jobs=-1,
             param_grid={'C': (0.01, 0.1, 1, 10, 100, 1000),
                         'gamma': ('scale', 'auto'),
                         'kernel': ('linear', 'poly', 'rbf', 'sigmoid')})

In [13]:
gs.best_params_


{'C': 1000, 'gamma': 'scale', 'kernel': 'linear'}

In [14]:
pred = gs.predict(X_test)
accuracy_score(y_test, pred)


0.83

In [15]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(500,), max_iter=2000).fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


0.8325

In [16]:
from sklearn.ensemble import GradientBoostingClassifier

clf = GradientBoostingClassifier(n_estimators=1000).fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


0.7975

In [17]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=2000).fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


0.8275

In [ ]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier().fit(X_train, y_train)
pred = clf.predict(X_test)


In [19]:
accuracy_score(y_test, pred)

0.825

In [ ]:
parameters = {
    "depth": [4, 5, 6, 7, 8, 9, 10],
    "learning_rate": [0.01, 0.02, 0.03, 0.04, 0.1],
    "iterations": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
}
clf = CatBoostClassifier()
grid_search_result = clf.grid_search(parameters, X=X_train, y=y_train)


In [21]:
grid_search_result["params"]

{'depth': 8, 'l2_leaf_reg': 1, 'iterations': 100, 'learning_rate': 0.1}

In [22]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(**grid_search_result["params"]).fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)

0:	learn: 1.3556536	total: 12.3ms	remaining: 1.22s
1:	learn: 1.1930172	total: 15.7ms	remaining: 771ms
2:	learn: 1.0748893	total: 17.9ms	remaining: 578ms
3:	learn: 0.9810387	total: 19.6ms	remaining: 471ms
4:	learn: 0.9062993	total: 22.3ms	remaining: 424ms
5:	learn: 0.8462163	total: 25.1ms	remaining: 393ms
6:	learn: 0.7994736	total: 27.7ms	remaining: 369ms
7:	learn: 0.7571785	total: 29.9ms	remaining: 344ms
8:	learn: 0.7194132	total: 32.7ms	remaining: 331ms
9:	learn: 0.6848448	total: 34.9ms	remaining: 314ms
10:	learn: 0.6562861	total: 37.1ms	remaining: 300ms
11:	learn: 0.6314848	total: 39.6ms	remaining: 291ms
12:	learn: 0.6107388	total: 41.8ms	remaining: 279ms
13:	learn: 0.5946732	total: 43.9ms	remaining: 270ms
14:	learn: 0.5781086	total: 46.2ms	remaining: 262ms
15:	learn: 0.5611974	total: 48.2ms	remaining: 253ms
16:	learn: 0.5473708	total: 50.9ms	remaining: 248ms
17:	learn: 0.5326800	total: 53.5ms	remaining: 244ms
18:	learn: 0.5227112	total: 55.8ms	remaining: 238ms
19:	learn: 0.5140844	t

0.8375